# Laboratorio · k-means en Colab con los socios de Diguillín

**Minería de Datos (IEI-097)** · IP Santo Tomás Chillán · Unidad II · Semana 10

El viernes agruparon **10 socios a mano**. Hoy agrupan a **todos** con el mismo algoritmo. El
código viene escrito y explicado: su trabajo es **ejecutarlo, entenderlo y decidir qué significa**.

| Celda | Qué trae |
|---|---|
| 🔎 **Cómo se resuelve** | Qué se pide, la idea y qué hace **cada línea** del código |
| **Código** | Para ejecutar (▶) y **leer** |
| ✔ **Verificación** | Compara con el número que debe salir |
| 🧭 **Qué debe salir** | Las líneas ✅ esperadas y qué hacer si sale otra cosa |
| ✍️ **Preguntas, bitácora y comparativa** | **En blanco: son de ustedes** |

**Antes de empezar:** `Archivo → Guardar una copia en Drive` (si no, sus respuestas se pierden).
Después ejecuten las celdas **en orden**, de arriba hacia abajo. Al final, `resumen()` debe
mostrar **14 de 14 verificaciones en ✅**.

**Integrantes:** …

## Celda 0 · Datos y limpieza

### 🔎 Qué hace la celda 0

Carga los datos y repite en pocas líneas la limpieza del trabajo de la Semana 4. Trae además `verificar()` y `resumen()`.

| Código | Qué hace |
|---|---|
| `import pandas as pd / import matplotlib.pyplot as plt` | las librerías de tablas y de gráficos. Colab ya trae pandas, matplotlib y scikit-learn: no hay que instalar nada |
| `pd.read_csv(RUTA_DATOS + "...")` | lee los CSV crudos de siempre desde el repositorio público: `clientes` y `boletas` |
| `drop_duplicates(subset="rut_norm", keep="first")` | los clientes duplicados por RUT normalizado, como en el trabajo de la Semana 4 |
| `imposible = nac.isna() ...` | las fechas de nacimiento imposibles: su edad queda vacía |
| `boletas[boletas.total >= 0]` | fuera las devoluciones |
| `boletas.fecha_hora.dt.year >= 2024` | fuera las boletas con la fecha falsa 1970-01-01 |
| `def verificar(...) / def resumen()` | las funciones de control de siempre: ✅ o ❌ contra el número esperado |

In [ ]:
# ============================================================
#  CELDA 0 · ejecutar primero, siempre
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)

# Los CSV crudos de siempre, desde internet
RUTA_DATOS = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/crudo/"
clientes = pd.read_csv(RUTA_DATOS + "clientes.csv")
boletas = pd.read_csv(RUTA_DATOS + "boletas.csv")

# ── La limpieza del trabajo de la Semana 4, en pocas líneas ────────────────────
clientes["rut_norm"] = (clientes.rut.str.replace(".", "", regex=False)
                                    .str.replace("-", "", regex=False).str.upper())
clientes = clientes.drop_duplicates(subset="rut_norm", keep="first").copy()
nac = pd.to_datetime(clientes.fecha_nacimiento, errors="coerce")
imposible = nac.isna() | (nac.dt.year < 1920) | (nac > pd.Timestamp("2025-06-30"))
clientes["edad"] = (2025 - nac.dt.year).where(~imposible)

boletas = boletas.drop_duplicates(subset="id_boleta", keep="first")
boletas = boletas[boletas.total >= 0].copy()                  # fuera las devoluciones
boletas["fecha_hora"] = pd.to_datetime(boletas.fecha_hora)
boletas = boletas[boletas.fecha_hora.dt.year >= 2024].copy()  # fuera la fecha falsa 1970-01-01

# Los números que deben salir en cada paso
ESPERADO = {
    'clientes_limpios': 1200,
    'boletas_limpias': 57372,
    'socios_sin_edad': 18,
    'socios_para_agrupar': 1182,
    'compras_mes_max': 8.8,
    'gasto_mil_max': 82.1,
    'grupo_mas_grande_crudo': 533,
    'edad_min_grupo_grande': 18,
    'edad_max_grupo_grande': 89,
    'tamanos_esc': [163, 293, 360, 366],
    'inercia_k3': 4550,
    'inercia_k4': 2975,
    'tamanos_final': [220, 294, 309, 359],
    'socios_en_grupo_del_nuevo': 220,
}
_resultados = {}

def verificar(clave, valor):
    """Compara su resultado con el esperado e imprime ✅ o ❌."""
    esperado = ESPERADO[clave]
    if hasattr(valor, "item"):
        valor = valor.item()
    ok = (valor == esperado)
    _resultados[clave] = ok
    print(("✅ " if ok else "❌ ") + f"{clave} = {valor}" + ("" if ok else f"   (debía salir {esperado})"))

def resumen():
    ok = sum(_resultados.values())
    print(f"{ok} de {len(ESPERADO)} verificaciones en ✅")
    pendientes = [k for k in ESPERADO if not _resultados.get(k)]
    if pendientes:
        print("Pendientes o con ❌:", ", ".join(pendientes))

verificar("clientes_limpios", len(clientes))
verificar("boletas_limpias", len(boletas))

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ clientes_limpios = 1200
✅ boletas_limpias = 57372
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| Error de conexión al leer un CSV | falló la descarga: ejecute la celda 0 de nuevo |
| `NameError` en cualquier celda | no ejecutó la celda 0 o una anterior: `Entorno de ejecución → Ejecutar anteriores` |

---
# Parte 0 · La tabla de socios

El viernes agruparon 10 socios a mano. Hoy son todos: primero hay que armar la tabla con una fila por socio.

## K1 · Una fila por socio

**Qué se pide.** k-means agrupa **filas**. Arme una tabla `socios` con una fila por socio y tres números: cuántas compras hace al mes, cuánto gasta por compra (en miles de pesos) y su edad.

### 🔎 Cómo se resuelve

**La idea.** Las boletas tienen una fila por compra; k-means necesita **una fila por socio**. Se agrupan las boletas por `id_cliente` (como una tabla dinámica) y se calculan dos números por socio. La edad se trae desde `clientes`. Los socios sin edad válida se dejan fuera, porque k-means no acepta celdas vacías.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `boletas[boletas.id_cliente.isin(clientes.id_cliente)]` | solo las boletas con un socio (las de clientes sin tarjeta no tienen `id_cliente`) |
| `groupby("id_cliente").agg(...)` | una fila por socio: cuenta sus boletas y promedia su total |
| `socios.compras / 12` | los datos cubren 12 meses: compras al mes |
| `socios.gasto / 1000` | el gasto por compra en **miles** de pesos, como en la actividad del viernes |
| `clientes.set_index("id_cliente").edad` | pandas pega la edad de cada socio buscándolo por su `id_cliente` (el BUSCARV de pandas) |
| `dropna(subset=["edad"])[[...]]` | fuera los socios sin edad y se quedan solo las tres columnas que se van a usar |

In [ ]:
# K1 · Una fila por socio

# Solo las boletas de socios (las que traen el id de un cliente limpio)
de_socios = boletas[boletas.id_cliente.isin(clientes.id_cliente)]

socios = de_socios.groupby("id_cliente").agg(
    compras=("id_boleta", "count"),
    gasto=("total", "mean"),
)
socios["compras_mes"] = (socios.compras / 12).round(1)   # julio 2024 a junio 2025 = 12 meses
socios["gasto_mil"] = (socios.gasto / 1000).round(1)     # gasto por compra, en miles de $
socios["edad"] = clientes.set_index("id_cliente").edad

socios_sin_edad = socios.edad.isna().sum()
print("Socios sin edad válida:", socios_sin_edad)
socios = socios.dropna(subset=["edad"])[["compras_mes", "gasto_mil", "edad"]]
print("Socios para agrupar:", len(socios))
socios.head()

In [ ]:
# ✔ Verificación · K1
verificar("socios_sin_edad", socios_sin_edad)
verificar("socios_para_agrupar", len(socios))

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ socios_sin_edad = 18
✅ socios_para_agrupar = 1182
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| Salen **1.200** socios y no 1.182 | faltó el `dropna`: quedaron los 18 socios sin edad, y k-means va a reclamar por los vacíos en la Parte III |
| `KeyError: 'id_cliente'` | no se ejecutó la celda 0, o se ejecutó dos veces una celda que cambia `socios`: ejecute de nuevo desde la celda 0 |

## K2 · Mirar los datos antes de agrupar

**Qué se pide.** Antes de correr un algoritmo, se mira. Resuma las tres columnas y dibuje a los socios en un gráfico de compras al mes contra gasto por compra.

### 🔎 Cómo se resuelve

**La idea.** `describe()` resume cada columna: promedio, mínimo, máximo y cuartiles. Mire los **rangos**: las compras al mes van de 0 a menos de 10; el gasto, de 6 a más de 80. Esa diferencia va a importar. El gráfico muestra si se ven grupos a ojo.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `socios.describe().round(1)` | el resumen de las tres columnas, con un decimal |
| `plt.figure(figsize=(7, 5))` | el tamaño del gráfico |
| `plt.scatter(x, y, s=8, alpha=0.5)` | un punto por socio, chico y semitransparente para ver dónde se amontonan |
| `plt.xlabel / plt.ylabel / plt.title` | los nombres de los ejes y el título |

In [ ]:
# K2 · Mirar los datos antes de agrupar

print(socios.describe().round(1))

plt.figure(figsize=(7, 5))
plt.scatter(socios.compras_mes, socios.gasto_mil, s=8, alpha=0.5)
plt.xlabel("Compras al mes")
plt.ylabel("Gasto por compra (miles de $)")
plt.title("Los socios de Diguillín")
plt.show()

In [ ]:
# ✔ Verificación · K2
verificar("compras_mes_max", socios.compras_mes.max())
verificar("gasto_mil_max", socios.gasto_mil.max())

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ compras_mes_max = 8.8
✅ gasto_mil_max = 82.1
```

> ✍️ **Bitácora · Parte 0 · La tabla de socios**
>
> - **Qué hicimos y qué entendimos:** …
> - **Fuente consultada** (documentación, compañeros, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Parte I · k-means con los datos tal como vienen

Dos variables, k = 4 y sin escalar: lo que haría alguien apurado.

## K3 · k-means con los datos tal como vienen

**Qué se pide.** La gerencia quiere **4 campañas**, así que pide **4 grupos**. Corra k-means con k = 4 sobre las compras al mes y el gasto por compra, **sin escalar**, y dibuje los grupos.

### 🔎 Cómo se resuelve

**La idea.** Es el mismo k-means del viernes, hecho por scikit-learn: elige 4 centroides, asigna cada socio al más cercano, mueve los centroides al promedio y repite hasta que nadie cambia. `n_init=10` lo corre **10 veces** con centroides iniciales distintos y se queda con el mejor; `random_state=2026` hace que a todos les salga lo mismo.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `from sklearn.cluster import KMeans` | trae k-means desde scikit-learn |
| `X = socios[["compras_mes", "gasto_mil"]]` | las dos variables, **sin escalar** |
| `KMeans(n_clusters=4, n_init=10, random_state=2026)` | k = 4, diez inicios, semilla fija |
| `km_crudo.fit_predict(X)` | **entrena** (hace las rondas) y devuelve el grupo de cada socio: 0, 1, 2 o 3 |
| `value_counts()` | cuántos socios quedaron en cada grupo |
| `km_crudo.cluster_centers_` | los 4 centroides: el «socio promedio» de cada grupo |
| `c=socios.grupo_crudo, cmap="tab10"` | pinta cada punto con el color de su grupo |
| `marker="X", s=200, c="black"` | los centroides, como X negras grandes |

In [ ]:
# K3 · k-means con los datos tal como vienen

from sklearn.cluster import KMeans

X = socios[["compras_mes", "gasto_mil"]]
km_crudo = KMeans(n_clusters=4, n_init=10, random_state=2026)
socios["grupo_crudo"] = km_crudo.fit_predict(X)
print(socios.grupo_crudo.value_counts())

centros = km_crudo.cluster_centers_
plt.figure(figsize=(7, 5))
plt.scatter(socios.compras_mes, socios.gasto_mil, c=socios.grupo_crudo, cmap="tab10", s=8)
plt.scatter(centros[:, 0], centros[:, 1], marker="X", s=200, c="black")
plt.xlabel("Compras al mes")
plt.ylabel("Gasto por compra (miles de $)")
plt.title("k = 4 sin escalar")
plt.show()

In [ ]:
# ✔ Verificación · K3
verificar("grupo_mas_grande_crudo", socios.grupo_crudo.value_counts().max())

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ grupo_mas_grande_crudo = 533
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| Los números de los grupos (0, 1, 2, 3) no coinciden con los de un compañero | es normal: el número es solo una etiqueta. Lo que tiene que coincidir son los **tamaños** |
| El grupo más grande no tiene 533 | cambió `n_init` o `random_state`, o se usaron otras columnas |

## K4 · El perfil de cada grupo

**Qué se pide.** Un grupo se entiende por su **perfil**: cuántos socios tiene y cómo es su socio promedio. Calcule el perfil de los 4 grupos, **incluida la edad** (que k-means no usó), y mire las edades de cada grupo en un gráfico de cajas.

### 🔎 Cómo se resuelve

**La idea.** El perfil es una tabla dinámica por grupo: cuántos socios y los promedios. Se agrega la **edad**, aunque k-means no la usó, para ver si los grupos tienen sentido. El gráfico de cajas muestra, para cada grupo, entre qué edades están sus socios.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `groupby("grupo_crudo").agg(...)` | una fila por grupo |
| `socios=("edad", "size")` | cuántos socios tiene el grupo |
| `compras_mes=("compras_mes", "mean")` | el promedio de cada variable: el perfil |
| `edad_min / edad_max` | el socio más joven y el más viejo del grupo |
| `perfil_crudo.socios.idxmax()` | **el número** del grupo con más socios |
| `perfil_crudo.loc[grande, "edad_min"]` | la edad mínima de ese grupo |
| `socios.boxplot(column="edad", by="grupo_crudo")` | una caja por grupo: la línea del medio es la mediana y la caja va del 25 % al 75 % de las edades |
| `plt.suptitle("")` | borra un título automático que pandas agrega arriba |

In [ ]:
# K4 · El perfil de cada grupo

perfil_crudo = socios.groupby("grupo_crudo").agg(
    socios=("edad", "size"),
    compras_mes=("compras_mes", "mean"),
    gasto_mil=("gasto_mil", "mean"),
    edad_prom=("edad", "mean"),
    edad_min=("edad", "min"),
    edad_max=("edad", "max"),
).round(1)
print(perfil_crudo)

grande = perfil_crudo.socios.idxmax()                  # el número del grupo más grande
edad_min_grande = perfil_crudo.loc[grande, "edad_min"]
edad_max_grande = perfil_crudo.loc[grande, "edad_max"]

socios.boxplot(column="edad", by="grupo_crudo", figsize=(7, 4))
plt.suptitle("")
plt.title("Edad de los socios de cada grupo (sin escalar)")
plt.show()

In [ ]:
# ✔ Verificación · K4
verificar("edad_min_grupo_grande", edad_min_grande)
verificar("edad_max_grupo_grande", edad_max_grande)

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ edad_min_grupo_grande = 18
✅ edad_max_grupo_grande = 89
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| `KeyError: 'grupo_crudo'` | no se ejecutó K3 |

### Preguntas · Parte I · k-means con los datos tal como vienen

Con sus palabras y **con los números** de su notebook.

**G1.** El grupo más grande junta socios de **18 a 89 años**. Mire su perfil: ¿en qué se parecen y en qué no? ¿Por qué k-means los juntó? *(Pista: compare el rango de las compras al mes con el del gasto en `describe()`).*

> ✍️ Su respuesta: …

> ✍️ **Bitácora · Parte I · k-means con los datos tal como vienen**
>
> - **Qué hicimos y qué entendimos:** …
> - **Fuente consultada** (documentación, compañeros, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Parte II · Escalar las variables

Lo mismo, con las dos variables en la escala de 0 a 10 de la cátedra.

## K5 · Escalar de 0 a 10 y repetir

**Qué se pide.** Lleve las dos variables a una escala de **0 a 10** (la fórmula de la cátedra) y corra k-means de nuevo con k = 4. Compare los grupos nuevos con los de la Parte I.

### 🔎 Cómo se resuelve

**La idea.** La distancia suma diferencias de variables distintas: la que se mide en números más grandes **manda**. `MinMaxScaler` aplica la fórmula de la cátedra, *(valor − mínimo) ÷ (máximo − mínimo) × 10*, a cada columna. Después es el mismo k-means. La tabla cruzada muestra adónde fueron los socios de cada grupo de la Parte I.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `MinMaxScaler(feature_range=(0, 10))` | la escala de 0 a 10: el mínimo queda en 0 y el máximo en 10 |
| `escala2.fit_transform(...)` | aprende el mínimo y el máximo de cada columna (`fit`) y los aplica (`transform`). Devuelve una tabla de números sin nombres |
| `sorted(socios.grupo_esc.value_counts().tolist())` | los tamaños de los grupos, de menor a mayor (así no importa el número de cada grupo) |
| `plt.scatter(socios.compras_mes, socios.gasto_mil, ...)` | el gráfico se dibuja con los valores **originales**, que se entienden; solo el color viene del modelo escalado |
| `pd.crosstab(socios.grupo_crudo, socios.grupo_esc)` | filas = grupo sin escalar, columnas = grupo escalado; cada celda cuenta socios |

In [ ]:
# K5 · Escalar de 0 a 10 y repetir

from sklearn.preprocessing import MinMaxScaler

escala2 = MinMaxScaler(feature_range=(0, 10))   # (valor − mín) ÷ (máx − mín) × 10
X_esc = escala2.fit_transform(socios[["compras_mes", "gasto_mil"]])

km_esc = KMeans(n_clusters=4, n_init=10, random_state=2026)
socios["grupo_esc"] = km_esc.fit_predict(X_esc)
tamanos_esc = sorted(socios.grupo_esc.value_counts().tolist())
print("Tamaños de los grupos:", tamanos_esc)

perfil_esc = socios.groupby("grupo_esc").agg(
    socios=("edad", "size"),
    compras_mes=("compras_mes", "mean"),
    gasto_mil=("gasto_mil", "mean"),
    edad_prom=("edad", "mean"),
    edad_min=("edad", "min"),
    edad_max=("edad", "max"),
).round(1)
print(perfil_esc)

plt.figure(figsize=(7, 5))
plt.scatter(socios.compras_mes, socios.gasto_mil, c=socios.grupo_esc, cmap="tab10", s=8)
plt.xlabel("Compras al mes")
plt.ylabel("Gasto por compra (miles de $)")
plt.title("k = 4 con las variables escaladas de 0 a 10")
plt.show()

# ¿Adónde fue cada grupo de la Parte I?
pd.crosstab(socios.grupo_crudo, socios.grupo_esc)

In [ ]:
# ✔ Verificación · K5
verificar("tamanos_esc", tamanos_esc)

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ tamanos_esc = [163, 293, 360, 366]
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| Los tamaños no coinciden | se escaló con otro rango (por ejemplo `StandardScaler` o 0 a 1 en vez de 0 a 10) o se escaló también la edad |

### Preguntas · Parte II · Escalar las variables

Con sus palabras y **con los números** de su notebook.

**G2.** ¿Qué cambió al escalar? Use la tabla cruzada y el perfil: ¿se separaron los jóvenes de los adultos mayores? ¿Por qué no del todo?

> ✍️ Su respuesta: …

> ✍️ **Bitácora · Parte II · Escalar las variables**
>
> - **Qué hicimos y qué entendimos:** …
> - **Fuente consultada** (documentación, compañeros, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Parte III · Agregar la edad, elegir k y usar el modelo

Una tercera variable, el método del codo, los grupos definitivos y un socio nuevo.

## K6 · Agregar la edad y buscar el codo

**Qué se pide.** Agregue la **edad** como tercera variable, escale las tres de 0 a 10 y corra k-means con k = 1, 2, … 8. Dibuje la **inercia** de cada k: ¿dónde se dobla la curva?

### 🔎 Cómo se resuelve

**La idea.** La edad separa a dos tipos de socio que compran parecido (jóvenes y mayores). Se escalan las tres variables y se prueba k-means con k = 1 a 8. La **inercia** (suma de las distancias² de cada socio a su centroide) siempre baja al agregar grupos: se busca el **codo**, donde deja de bajar mucho.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `variables = [...]` | ahora son tres columnas |
| `escala3.fit_transform(socios[variables])` | las tres, de 0 a 10 |
| `for k in range(1, 9):` | repite para k = 1, 2, … 8 |
| `KMeans(...).fit(X3)` | entrena un k-means con ese k |
| `round(km.inertia_)` | su inercia, redondeada, se agrega a la lista |
| `plt.plot(range(1, 9), inercias, marker="o")` | la curva del codo |

In [ ]:
# K6 · Agregar la edad y buscar el codo

variables = ["compras_mes", "gasto_mil", "edad"]
escala3 = MinMaxScaler(feature_range=(0, 10))
X3 = escala3.fit_transform(socios[variables])

inercias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=2026).fit(X3)
    inercias.append(round(km.inertia_))
print("Inercia para k = 1 … 8:", inercias)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 9), inercias, marker="o")
plt.xlabel("k (cantidad de grupos)")
plt.ylabel("Inercia")
plt.title("Método del codo · tres variables escaladas")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ✔ Verificación · K6
verificar("inercia_k3", inercias[2])
verificar("inercia_k4", inercias[3])

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ inercia_k3 = 4550
✅ inercia_k4 = 2975
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| `ValueError: Input X contains NaN` | quedaron socios sin edad: falta el `dropna` de K1 |
| La celda tarda | es normal: son 8 k-means de 10 inicios cada uno (unos segundos) |

## K7 · Los cuatro grupos definitivos

**Qué se pide.** Con las tres variables escaladas, corra k-means con k = 4. Calcule el perfil de cada grupo y dibújelos en dos gráficos: compras contra gasto, y edad contra gasto.

### 🔎 Cómo se resuelve

**La idea.** Los grupos definitivos: k = 4 con las tres variables escaladas. El perfil se calcula con los valores **originales** para poder leerlo («25 años, 2,5 compras al mes»). Dos gráficos, porque con tres variables no basta uno.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `km4.fit_predict(X3)` | los grupos definitivos |
| `plt.subplots(1, 2, figsize=(12, 4.5))` | dos gráficos lado a lado: `ejes[0]` y `ejes[1]` |
| `ejes[1].scatter(socios.edad, socios.gasto_mil, ...)` | el segundo gráfico usa la edad en el eje x |
| `fig.suptitle(...)` | un título para los dos |

In [ ]:
# K7 · Los cuatro grupos definitivos

km4 = KMeans(n_clusters=4, n_init=10, random_state=2026)
socios["grupo"] = km4.fit_predict(X3)
tamanos_final = sorted(socios.grupo.value_counts().tolist())
print("Tamaños de los grupos:", tamanos_final)

perfil = socios.groupby("grupo").agg(
    socios=("edad", "size"),
    compras_mes=("compras_mes", "mean"),
    gasto_mil=("gasto_mil", "mean"),
    edad_prom=("edad", "mean"),
    edad_min=("edad", "min"),
    edad_max=("edad", "max"),
).round(1)
print(perfil)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
ejes[0].scatter(socios.compras_mes, socios.gasto_mil, c=socios.grupo, cmap="tab10", s=8)
ejes[0].set_xlabel("Compras al mes")
ejes[0].set_ylabel("Gasto por compra (miles de $)")
ejes[1].scatter(socios.edad, socios.gasto_mil, c=socios.grupo, cmap="tab10", s=8)
ejes[1].set_xlabel("Edad")
ejes[1].set_ylabel("Gasto por compra (miles de $)")
fig.suptitle("k = 4 con compras, gasto y edad (escaladas)")
plt.show()

In [ ]:
# ✔ Verificación · K7
verificar("tamanos_final", tamanos_final)

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ tamanos_final = [220, 294, 309, 359]
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| Los tamaños no coinciden | revise que `X3` venga de K6 con las tres variables |

## K8 · Un socio nuevo

**Qué se pide.** Llega un socio nuevo: compra **6 veces al mes**, gasta **$14 mil** por compra y tiene **78 años**. ¿A qué grupo lo asigna el modelo? Después cambie los números por un socio inventado por usted.

### 🔎 Cómo se resuelve

**La idea.** Un modelo de clustering también **clasifica casos nuevos**: el socio nuevo va al centroide más cercano. Antes hay que **escalarlo con la misma escala** de K6 (`transform`, sin `fit`): si no, sus números no son comparables.

**Línea por línea:**

| Código | Qué hace |
|---|---|
| `pd.DataFrame({...})` | el socio nuevo como tabla de una fila, con las mismas columnas |
| `escala3.transform(nuevo)` | lo escala con el mínimo y el máximo de **los socios** (no los recalcula) |
| `km4.predict(...)[0]` | el grupo del centroide más cercano; `[0]` toma el primero (y único) resultado |
| `perfil.loc[grupo_nuevo]` | el perfil de ese grupo, para ver si tiene sentido |

In [ ]:
# K8 · Un socio nuevo

nuevo = pd.DataFrame({"compras_mes": [6.0], "gasto_mil": [14.0], "edad": [78]})

grupo_nuevo = km4.predict(escala3.transform(nuevo))[0]   # se escala con la MISMA escala
print("El socio nuevo va al grupo", grupo_nuevo)
perfil.loc[grupo_nuevo]

In [ ]:
# ✔ Verificación · K8
verificar("socios_en_grupo_del_nuevo", perfil.loc[grupo_nuevo, "socios"])

### 🧭 Qué debe salir

La verificación de arriba tiene que imprimir:

```
✅ socios_en_grupo_del_nuevo = 220
```

**Si sale otra cosa:**

| Síntoma | Causa y arreglo |
|---|---|
| `ValueError: X has 2 features, but MinMaxScaler is expecting 3` | falta una columna en `nuevo`: tienen que ser las tres, con los mismos nombres |
| El socio va a un grupo que no tiene sentido | ¿usó `fit_transform` en vez de `transform`? Eso escala al socio contra sí mismo |

### Preguntas · Parte III · Agregar la edad, elegir k y usar el modelo

Con sus palabras y **con los números** de su notebook.

**G3.** ¿En qué k se dobla el codo? La gerencia pidió 4 grupos: ¿lo respaldan los datos? ¿Qué dos grupos se juntarían con k = 3, y qué se perdería?

> ✍️ Su respuesta: …

**G4.** Póngale **nombre** a cada uno de los 4 grupos (mire su perfil) y proponga **una campaña** distinta para cada uno.

> ✍️ Su respuesta: …

**G5.** ¿A qué grupo fue el socio nuevo? ¿Tiene sentido? ¿Qué socio inventó usted, a qué grupo fue y por qué?

> ✍️ Su respuesta: …

> ✍️ **Bitácora · Parte III · Agregar la edad, elegir k y usar el modelo**
>
> - **Qué hicimos y qué entendimos:** …
> - **Fuente consultada** (documentación, compañeros, IA — cuál y para qué): …
> - **Qué no funcionó a la primera:** …

---
# Cierre · A mano y en Colab, lado a lado

Complete la tabla (doble clic en esta celda para editarla).

| | A mano (vie 09/10) | Colab (hoy) |
|---|---|---|
| Cuántos socios | | |
| Variables | | |
| k | | |
| Centroides iniciales | | |
| Escalar | | |
| Rondas hasta terminar | | |
| Cómo se eligió k | | |

In [ ]:
# ✔ Resumen final
resumen()

*Supermercados Diguillín es una empresa ficticia; los datos son sintéticos.*